# CS 7150 Final Project - Phase 1: Baseline MLP

Binary patch classification: cell vs background

Concepts from Lecture 4:
- Cross-entropy loss
- MLP architecture
- SGD + Momentum
- Learning rate scheduling

In [26]:
import sys
sys.path.insert(0, "..")
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import PanNukeDataset
from src.models.mlp import PatchMLP

# Load and Explore Data

In [27]:
# Load dataset
train_ds = PanNukeDataset(split="fold1")
val_ds = PanNukeDataset(split="fold2")
test_ds = PanNukeDataset(split="fold3")

print(f"Train size: {len(train_ds)}")
print(f"Val size: {len(val_ds)}")
print(f"Test size: {len(test_ds)}")

Train size: 2656
Val size: 2523
Test size: 2722


In [28]:
# Manual implementation of a dense (fully connected) layer
# Manual gradient computation without autograd
# Demonstration of backpropagation 
class ManualDenseLayer:
    def __init__(self, in_features, out_features):
        # Use PyTorch for tensor ops
        self.W = torch.randn(in_features, out_features) * 0.01 
        # if all weights are 0, no learning happens
        # if all weights are equal, all neurons learn the same features (symmetry problem)
        # if weights are too large, activations can explode
        
        self.b = torch.zeros(out_features)
        # You'll store these for backward pass
        self.x_cache = None
        
    # computes y = xW + b    
    def forward(self, x):
        self.x_cache = x  # Cache for backward
        return x @ self.W + self.b
    
    def backward(self, grad_output):
        # grad_output = ∂L/∂y (gradient flowing back from the next layer)
        self.grad_W = self.x_cache.T @ grad_output
        # (in_features, batch_size) @ (batch_size, out_features) = (in_features, out_features)
        
        self.grad_b = grad_output.sum(dim=0)
        # Sum over batch dimension → shape (out_features,)
        
        grad_input = grad_output @ self.W.T
        # (batch_size, out_features) @ (out_features, in_features) = (batch_size, in_features)

        return grad_input

In [29]:
class ManualReLULayer:
    def forward(self, x):
        self.mask = (x > 0)  # Record whether or not to let x through - we don't need to cache x itself for ReLU because the derivative is either 0 or 1
        return x * self.mask
        
    def backward(self, grad_output):
        return grad_output * self.mask

In [30]:
class ManualCrossEntropyLoss:
    def forward(self, logits, targets):
        # Step 1: Softmax
        # Hint: subtract max for stability, then exp, then divide by sum
        logits_stable = logits - torch.max(logits, dim=1, keepdim=True).values
        exps = torch.exp(logits_stable)
        self.probs = exps / torch.sum(exps, dim=1, keepdim=True)
        
        # Step 2: Negative log likelihood
        batch_size = logits.shape[0]
        correct_probs = self.probs[torch.arange(batch_size), targets]
        loss = -torch.log(correct_probs).mean()
        
        # Cache for backward
        self.targets = targets
        self.batch_size = batch_size
        
        return loss
    
    def backward(self):
        # Gradient of softmax + cross-entropy combined
        grad_logits = self.probs.clone()
        grad_logits[torch.arange(self.batch_size), self.targets] -= 1
        grad_logits /= self.batch_size  # Average over batch
        
        return grad_logits

In [31]:
class ManualSGD:
    def __init__(self, layers, lr=0.01):
        self.layers = layers
        self.lr = lr
    
    def step(self):
        for layer in self.layers:
            layer.W = layer.W - self.lr * layer.grad_W
            layer.b = layer.b - self.lr * layer.grad_b